# CS156 · Session 8 — Gradients: Multivariate Derivatives

This notebook reconstructs the supplied **Generated Forum Code Workbook** for the gradients session.

The session focuses on computing and using gradients, including:
- scalar backpropagation through a sigmoid computation graph,
- multivariate derivatives for a dot product,
- vectorized backpropagation,
- automatic differentiation with JAX,
- an extension using gradient descent to tune weights.

Run the cells **top to bottom**.

## 1. Basic prep — sigmoid derivative

The workbook begins by recalling the sigmoid

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

and its derivative

$$
\sigma'(x)=\sigma(x)(1-\sigma(x)).
$$

This derivative is the local gradient used when backpropagating through a sigmoid.

## 2. Scalar forward and backward pass

The first coding exercise computes a scalar sigmoid forward pass from two weights and two inputs, then manually applies the chain rule backward from the final output.

The workbook emphasizes that intermediate forward-pass values must be stored because they are reused during backpropagation.

In [ ]:
import numpy as np

# Forward pass
w0, w1, w2 = 2.00, -3.00, -3.00
x0, x1 = -1.00, -2.00

mul0 = w0 * x0      # -2.00
mul1 = w1 * x1      #  6.00
sum01 = mul0 + mul1 #  4.00
sum012 = sum01 + w2 #  1.00

neg = -sum012       # -1.00
exp = np.exp(neg)   #  0.37
plus1 = exp + 1     #  1.37
invert = 1 / plus1  #  0.73

# Backward pass
# All derivatives are with respect to the final output "invert".
d_invert = 1.00
d_plus1 = d_invert * (-1 / plus1**2)
d_exp = d_plus1 * 1
d_neg = d_exp * exp
d_sum012 = d_neg * (-1)

# Addition sends the same upstream gradient to each input.
d_w2, d_sum01 = d_sum012, d_sum012
d_mul0, d_mul1 = d_sum01, d_sum01

# Multiplication: derivative with respect to one input is the other input.
d_w1, d_x1 = d_mul1 * x1, d_mul1 * w1
d_w0, d_x0 = d_mul0 * x0, d_mul0 * w0

print("sigmoid output:", invert)
print("d_w0, d_w1, d_w2:", d_w0, d_w1, d_w2)
print("d_x0, d_x1:", d_x0, d_x1)

## 3. Calculus derivation — multivariate dot product

For

$$
w=
\begin{bmatrix}
w_1\\w_2\\w_3
\end{bmatrix},
\qquad
x=
\begin{bmatrix}
x_1\\x_2\\x_3
\end{bmatrix},
\qquad
f(w,x)=w^Tx=w_1x_1+w_2x_2+w_3x_3,
$$

the workbook gives

$$
\frac{df}{dx}
=
\begin{bmatrix}
w_1\\w_2\\w_3
\end{bmatrix}
=w.
$$

Differentiating with respect to the components of $w$ instead gives

$$
\frac{df}{dw}
=
\begin{bmatrix}
x_1\\x_2\\x_3
\end{bmatrix}
=x.
$$

Each $w_i$ appears only in the term $w_i x_i$, so differentiating with respect to $w_i$ leaves $x_i$.

## 4. Vectorized sigmoid backpropagation

The next exercise fuses the scalar multiply-and-add operations into the vector dot product $w^Tx$.

The underlying calculus is unchanged; only the implementation becomes vectorized.

In [ ]:
# Forward pass
w = np.array([2.00, -3.00])
b = -3.00
x = np.array([-1.00, -2.00])

dot = w.T @ x       # 4.00
summed = dot + b    # 1.00
neg = -summed       # -1.00
exp = np.exp(neg)   # 0.37
plus1 = exp + 1     # 1.37
invert = 1 / plus1  # 0.73

# Backward pass
d_invert = 1.00

d_plus1 = d_invert * (-1 / plus1**2)
d_exp = d_plus1
d_neg = d_exp * exp
d_summed = d_neg * (-1)

d_b, d_dot = d_summed, d_summed

# These are vectors.
d_w = d_dot * x
d_x = d_dot * w

print("sigmoid output:", invert)
print("d_w:", d_w)
print("d_b:", d_b)
print("d_x:", d_x)

## 5. Preview — automatic differentiation with JAX

The workbook introduces JAX as a NumPy-like library that can automatically compute the backward pass.

The original Forum workbook included an installation cell for a specific JAX/JAXLIB version. In a local environment, install JAX before running the next cells if it is not already available.

In [ ]:
import jax
import jax.numpy as jnp

def sigmoid_example(params, x):
    dot = params["w"].T @ x
    summed = dot + params["b"]

    neg = -summed
    exp = jnp.exp(neg)
    plus1 = exp + 1
    invert = 1 / plus1

    return invert

In [ ]:
params = {
    "w": jnp.array([2.00, -3.00]),
    "b": -3.00,
}
x = jnp.array([-1.00, -2.00])

sigmoid_example(params, x)

In [ ]:
grad_sigmoid_example = jax.grad(sigmoid_example)
grad_sigmoid_example(params, x)

In the workbook's recorded run, JAX returns gradients approximately equal to:

- $\partial y/\partial b \approx 0.1966$
- $\partial y/\partial w \approx [-0.1966, -0.3932]$

These match the manually derived vectorized gradients.

## 6. Extension — gradient descent on the weights

The extension asks for a function that repeatedly updates the weights so the sigmoid output becomes close to 0.

The workbook keeps $x$ and $b$ fixed and performs 1000 gradient-descent iterations on $w$.

In [ ]:
import numpy as np

x = np.array([-1.00, -2.00])
b = -3.00
w = np.array([2.00, -3.00])

def sigmoid(w, x, b):
    z = w.T @ x + b
    return 1 / (1 + np.exp(-z))

def gradient(w, x, b):
    y = sigmoid(w, x, b)

    # Derivative of sigmoid output with respect to w
    return y * (1 - y) * x

def gradient_descent(w, x, b, lr=0.1, iterations=1000):
    w = w.copy()

    for _ in range(iterations):
        grad = gradient(w, x, b)
        w = w - lr * grad

    return w

w_final = gradient_descent(w, x, b)

print("Final weights:", w_final)
print("Final sigmoid output:", sigmoid(w_final, x, b))

### Recorded extension result

The workbook's run reported approximately:

- **Final weights:** $[3.43659368, -0.12681264]$
- **Final sigmoid output:** $0.0020600386$

The goal of the extension is to connect the gradient calculation to optimization: repeatedly stepping in the negative gradient direction changes the parameters so the output moves toward the chosen objective.